# Tech Challenge Fase 3 — Modelagem Supervisionada

## Objetivo

Treinar um modelo de classificação **simples, reproduzível e explicável** para prever se um aluno será alfabetizado.

O modelo principal será uma **Regressão Logística**.

### Estratégia temporal

- **2023** → desenvolvimento e seleção de hiperparâmetros;
- **2024** → teste final *out-of-time*.

Para manter compatibilidade com Databricks Serverless, a seleção de regularização será feita com uma divisão explícita **80/20 dentro de 2023**, sem usar validação cruzada distribuída.

### Features

**Categóricas**
- `rede`
- `sigla_uf`
- `regiao`

**Numéricas**
- `log_populacao`
- `vinculos_ativos_por_1000_habitantes`
- `proporcao_vinculos_estatutarios`


## 1. Configurações


In [ ]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import pandas as pd

CATALOG = "workspace"
GOLD_SCHEMA = "alfabetizacao_gold"

MODEL_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.base_modelagem_aluno"
METRICS_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.metricas_modelo_fase3"

SEED = 42
TUNING_TARGET_PER_CLASS = 200_000
REG_PARAMS = [0.0, 0.01, 0.1]

print("Base:", MODEL_TABLE)
print("Seed:", SEED)
print("Regularizações testadas:", REG_PARAMS)


## 2. Leitura da Gold


In [ ]:
df_raw = spark.table(MODEL_TABLE)

display(
    df_raw
    .groupBy("ano", "grupo_modelagem")
    .agg(
        F.count("*").alias("registros"),
        F.round(
            F.avg("target_alfabetizado") * 100,
            2,
        ).alias("percentual_alfabetizados"),
    )
    .orderBy("ano")
)


## 3. Feature engineering

São criadas duas features derivadas:

### `log_populacao`

A população possui distribuição muito assimétrica. O logaritmo reduz o impacto de municípios extremamente grandes.

### `proporcao_vinculos_estatutarios`

Em vez de utilizar novamente um volume absoluto altamente correlacionado com população e vínculos ativos, utilizamos a proporção:

`quantidade_vinculos_estatutarios / quantidade_vinculos_ativos`

Também é adicionada a região geográfica da UF para melhorar a generalização territorial, inclusive para UFs ausentes no desenvolvimento de 2023.


In [ ]:
regiao_map = F.create_map(
    *[
        F.lit(x)
        for x in [
            "AC", "Norte",
            "AP", "Norte",
            "AM", "Norte",
            "PA", "Norte",
            "RO", "Norte",
            "RR", "Norte",
            "TO", "Norte",

            "AL", "Nordeste",
            "BA", "Nordeste",
            "CE", "Nordeste",
            "MA", "Nordeste",
            "PB", "Nordeste",
            "PE", "Nordeste",
            "PI", "Nordeste",
            "RN", "Nordeste",
            "SE", "Nordeste",

            "DF", "Centro-Oeste",
            "GO", "Centro-Oeste",
            "MT", "Centro-Oeste",
            "MS", "Centro-Oeste",

            "ES", "Sudeste",
            "MG", "Sudeste",
            "RJ", "Sudeste",
            "SP", "Sudeste",

            "PR", "Sul",
            "RS", "Sul",
            "SC", "Sul",
        ]
    ]
)

df = (
    df_raw
    .withColumn(
        "rede",
        F.col("rede").cast("string"),
    )
    .withColumn(
        "regiao",
        regiao_map[F.col("sigla_uf")],
    )
    .withColumn(
        "log_populacao",
        F.log1p(F.col("populacao").cast("double")),
    )
    .withColumn(
        "proporcao_vinculos_estatutarios",
        F.when(
            F.col("quantidade_vinculos_ativos") > 0,
            (
                F.col("quantidade_vinculos_estatutarios")
                / F.col("quantidade_vinculos_ativos")
            ),
        ).otherwise(F.lit(0.0)),
    )
)

feature_cols = [
    "rede",
    "sigla_uf",
    "regiao",
    "log_populacao",
    "vinculos_ativos_por_1000_habitantes",
    "proporcao_vinculos_estatutarios",
]

display(
    df.select(
        *feature_cols,
        "target_alfabetizado",
        "ano",
    ).limit(10)
)


## 4. Validação das features

Nenhuma feature utilizada pelo modelo deve possuir valores ausentes.


In [ ]:
expressoes_nulos = [
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(f"{c}_nulos")
    for c in feature_cols
]

display(
    df
    .groupBy("ano")
    .agg(*expressoes_nulos)
    .orderBy("ano")
)


## 5. Separação temporal

O conjunto de desenvolvimento contém somente 2023.

O teste final contém somente 2024.

> **Compatibilidade Serverless:** este notebook não utiliza `cache()`/`persist()`, pois essas operações não são suportadas no Databricks Serverless deste ambiente.


In [ ]:
df_dev = (
    df
    .filter(F.col("grupo_modelagem") == "DESENVOLVIMENTO_2023")
)

df_test = (
    df
    .filter(F.col("grupo_modelagem") == "TESTE_TEMPORAL_2024")
)

print("Desenvolvimento 2023:", df_dev.count())
print("Teste temporal 2024:", df_test.count())


## 6. Amostra estratificada para seleção do hiperparâmetro

A seleção de regularização será feita em uma amostra estratificada de aproximadamente 400 mil registros de 2023.

Depois, essa amostra será dividida em **80% treino** e **20% validação**. O ano de 2024 permanece totalmente isolado.


In [ ]:
contagens = {
    int(row["target_alfabetizado"]): int(row["count"])
    for row in (
        df_dev
        .groupBy("target_alfabetizado")
        .count()
        .collect()
    )
}

fractions = {
    classe: min(
        1.0,
        TUNING_TARGET_PER_CLASS / quantidade,
    )
    for classe, quantidade in contagens.items()
}

print("Contagens por classe:", contagens)
print("Frações de amostragem:", fractions)

df_tuning = (
    df_dev
    .sampleBy(
        "target_alfabetizado",
        fractions=fractions,
        seed=SEED,
    )
)

display(
    df_tuning
    .groupBy("target_alfabetizado")
    .count()
    .orderBy("target_alfabetizado")
)

print("Total da amostra de tuning:", df_tuning.count())


## 7. Pipeline de preprocessing

O preprocessing será integrado ao modelo para evitar inconsistência entre treino e teste.

### Categóricas
- `StringIndexer`
- `OneHotEncoder`
- categorias não vistas no treino são tratadas com `handleInvalid="keep"`

### Numéricas
- montagem em vetor
- padronização por desvio-padrão

### Modelo
- Regressão Logística com regularização L2


In [ ]:
categorical_cols = [
    "rede",
    "sigla_uf",
    "regiao",
]

numeric_cols = [
    "log_populacao",
    "vinculos_ativos_por_1000_habitantes",
    "proporcao_vinculos_estatutarios",
]


def build_pipeline(reg_param):
    indexers = [
        StringIndexer(
            inputCol=coluna,
            outputCol=f"{coluna}_idx",
            handleInvalid="keep",
        )
        for coluna in categorical_cols
    ]

    encoder = OneHotEncoder(
        inputCols=[f"{c}_idx" for c in categorical_cols],
        outputCols=[f"{c}_ohe" for c in categorical_cols],
        handleInvalid="keep",
    )

    numeric_assembler = VectorAssembler(
        inputCols=numeric_cols,
        outputCol="numeric_features",
    )

    numeric_scaler = StandardScaler(
        inputCol="numeric_features",
        outputCol="numeric_scaled",
        withStd=True,
        withMean=False,
    )

    final_assembler = VectorAssembler(
        inputCols=["rede_ohe", "sigla_uf_ohe", "regiao_ohe", "numeric_scaled"],
        outputCol="features",
    )

    lr = LogisticRegression(
        labelCol="target_alfabetizado",
        featuresCol="features",
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        maxIter=50,
        elasticNetParam=0.0,
        regParam=float(reg_param),
        standardization=False,
    )

    return Pipeline(stages=[*indexers, encoder, numeric_assembler, numeric_scaler, final_assembler, lr])


## 8. Divisão treino × validação dentro de 2023

A amostra estratificada é dividida em 80% treino e 20% validação com seed fixo.


In [ ]:
df_train_tuning, df_val_tuning = df_tuning.randomSplit(
    [0.8, 0.2],
    seed=SEED,
)

print("Treino tuning:", df_train_tuning.count())
print("Validação tuning:", df_val_tuning.count())

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="target_alfabetizado",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)


## 9. Seleção simples da regularização

São testados apenas três valores de `regParam`. Cada candidato é treinado no conjunto de treino de 2023 e avaliado no conjunto de validação de 2023. A métrica de seleção é ROC-AUC.


In [ ]:
tuning_results = []

for reg_param in REG_PARAMS:
    print(f"Treinando regParam={reg_param}...")

    candidate_pipeline = build_pipeline(reg_param)
    candidate_model = candidate_pipeline.fit(df_train_tuning)
    candidate_pred = candidate_model.transform(df_val_tuning)

    auc = auc_evaluator.evaluate(candidate_pred)
    tuning_results.append((float(reg_param), float(auc)))

    print(f"ROC-AUC validação: {auc:.6f}")

df_tuning_results = spark.createDataFrame(
    tuning_results,
    ["reg_param", "roc_auc_validacao"],
)

display(df_tuning_results.orderBy(F.desc("roc_auc_validacao")))


In [ ]:
best_reg_param, best_validation_auc = max(
    tuning_results,
    key=lambda x: x[1],
)

print("Melhor regParam:", best_reg_param)
print("Melhor ROC-AUC de validação:", best_validation_auc)


## 10. Treinamento final em todo o ano de 2023

Após a seleção do hiperparâmetro, o modelo é treinado novamente usando **todos os registros de 2023**.


In [ ]:
final_pipeline = build_pipeline(best_reg_param)
final_model = final_pipeline.fit(df_dev)

print("Modelo final treinado em todo o ano de 2023.")


## 11. Avaliação final em 2024

Agora o modelo é aplicado uma única vez ao teste temporal completo de 2024.


In [ ]:
pred_test = (
    final_model
    .transform(df_test)
)

print("Registros avaliados:", pred_test.count())


## 12. Matriz de confusão e métricas

A classe positiva é:

`1 = alfabetizado`


In [ ]:
confusao = (
    pred_test
    .agg(
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 1)
                & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("tp"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 0)
                & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("tn"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 0)
                & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("fp"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 1)
                & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("fn"),
    )
    .first()
)

tp = int(confusao["tp"])
tn = int(confusao["tn"])
fp = int(confusao["fp"])
fn = int(confusao["fn"])

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else 0.0
)

roc_auc = auc_evaluator.evaluate(pred_test)

metricas = [
    ("accuracy", float(accuracy)),
    ("precision", float(precision)),
    ("recall", float(recall)),
    ("f1", float(f1)),
    ("roc_auc", float(roc_auc)),
]

display(
    spark.createDataFrame(
        metricas,
        ["metrica", "valor"],
    )
)

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)


## 13. Baseline simples

Como o conjunto de desenvolvimento de 2023 possui ligeira maioria de alfabetizados, o baseline mais simples é prever sempre a classe `1`.

O objetivo é confirmar que o modelo realmente agrega informação além da classe majoritária.


In [ ]:
majority_class = (
    df_dev
    .groupBy("target_alfabetizado")
    .count()
    .orderBy(F.desc("count"))
    .first()["target_alfabetizado"]
)

print("Classe majoritária em 2023:", majority_class)

baseline_test = (
    df_test
    .withColumn(
        "prediction_baseline",
        F.lit(float(majority_class)),
    )
)

baseline_conf = (
    baseline_test
    .agg(
        F.sum(
            F.when(
                F.col("target_alfabetizado")
                == F.col("prediction_baseline"),
                1,
            ).otherwise(0)
        ).alias("acertos"),
        F.count("*").alias("total"),
    )
    .first()
)

baseline_accuracy = (
    baseline_conf["acertos"]
    / baseline_conf["total"]
)

print(
    "Accuracy baseline em 2024:",
    round(baseline_accuracy, 4),
)


## 14. Coeficientes do modelo

A Regressão Logística permite inspecionar diretamente seus coeficientes.

- coeficiente positivo → aumenta a tendência de classificação como alfabetizado;
- coeficiente negativo → reduz essa tendência;
- magnitude maior → efeito mais forte na escala transformada.

A interpretação é associativa, não causal.


In [ ]:
lr_model = final_model.stages[-1]

coeficientes = lr_model.coefficients.toArray()

print("Intercepto:", lr_model.intercept)
print("Quantidade de coeficientes:", len(coeficientes))


In [ ]:
# Tenta recuperar os nomes das features a partir dos metadados
# produzidos pelo VectorAssembler.

sample_transformed = (
    final_model
    .transform(df_dev.limit(1))
)

metadata = (
    sample_transformed
    .schema["features"]
    .metadata
)

feature_names = []

try:
    attrs = metadata["ml_attr"]["attrs"]

    flattened = []

    for attr_type in attrs:
        flattened.extend(attrs[attr_type])

    flattened = sorted(
        flattened,
        key=lambda x: x["idx"],
    )

    feature_names = [
        item["name"]
        for item in flattened
    ]

except Exception as exc:
    print(
        "Não foi possível recuperar automaticamente "
        "todos os nomes das features:",
        exc,
    )

if len(feature_names) == len(coeficientes):
    coef_df = pd.DataFrame(
        {
            "feature": feature_names,
            "coeficiente": coeficientes,
        }
    )

    coef_df["abs_coeficiente"] = (
        coef_df["coeficiente"].abs()
    )

    coef_df = (
        coef_df
        .sort_values(
            "abs_coeficiente",
            ascending=False,
        )
    )

    display(coef_df.head(30))

else:
    print(
        "Coeficientes disponíveis, mas os nomes "
        "das features não foram recuperados integralmente."
    )


## 15. Persistência das métricas

As principais métricas do teste temporal serão registradas em uma tabela Gold.


In [ ]:
metricas_output = [
    (
        "logistic_regression",
        2023,
        2024,
        best_reg_param,
        "validation_roc_auc",
        float(best_validation_auc),
    ),
    (
        "logistic_regression",
        2023,
        2024,
        best_reg_param,
        "accuracy",
        float(accuracy),
    ),
    (
        "logistic_regression",
        2023,
        2024,
        best_reg_param,
        "precision",
        float(precision),
    ),
    (
        "logistic_regression",
        2023,
        2024,
        best_reg_param,
        "recall",
        float(recall),
    ),
    (
        "logistic_regression",
        2023,
        2024,
        best_reg_param,
        "f1",
        float(f1),
    ),
    (
        "logistic_regression",
        2023,
        2024,
        best_reg_param,
        "roc_auc",
        float(roc_auc),
    ),
    (
        "baseline_majority",
        2023,
        2024,
        None,
        "accuracy",
        float(baseline_accuracy),
    ),
]

df_metricas_output = (
    spark.createDataFrame(
        metricas_output,
        [
            "modelo",
            "ano_treino",
            "ano_teste",
            "reg_param",
            "metrica",
            "valor",
        ],
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp(),
    )
)

(
    df_metricas_output
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(METRICS_TABLE)
)

display(
    spark.table(METRICS_TABLE)
)


## 16. Conclusão

O modelo foi mantido propositalmente simples e explicável.

### Modelo principal

**Regressão Logística**

### Estratégia

- amostra estratificada de 2023 para seleção de regularização;
- divisão explícita 80/20 entre treino e validação;
- comparação de três níveis de regularização;
- treinamento final em todo 2023;
- teste final exclusivamente em 2024;
- preprocessing integrado em uma única Pipeline.

Essa abordagem evita dependência de mecanismos internos de cache da validação cruzada distribuída no Databricks Serverless.
